The following notebook structure is mostly LLM generated to get a quick compatibility with the built multi-model config system.

In [31]:
from __future__ import annotations

from pathlib import Path
import pickle
import yaml
import numpy as np
import pandas as pd
import torch
import soundfile as sf
from tqdm import tqdm

from dcase.src.cnn_experiment import CNNExperiment
from dcase.src.models import EnsembleCNNModel, SklearnAudioEnsembleClassifier, CNNTCNModel
from models import BaselineModel, LinSeqModel, CNNModel, SklearnAudioClassifier
from baseline_experiment import BaselineExperiment

In [32]:
MODEL_REGISTRY = {
    "BaselineModel": BaselineModel,
    "LinSeqModel": LinSeqModel,
    "CNNModel": CNNModel,
    "CNNEnsemble": EnsembleCNNModel,
    "EnsembleCNNModel": EnsembleCNNModel,
    "SklearnAudioClassifier": SklearnAudioClassifier,
    "SklearnAudioEnsembleClassifier": SklearnAudioEnsembleClassifier,
    "CNNTCNModel": CNNTCNModel,
}

EXPERIMENT_REGISTRY = {
    "BaselineModel": BaselineExperiment,
    "LinSeqModel": BaselineExperiment,
    "CNNModel": CNNExperiment,
    "CNNEnsemble": CNNExperiment,
    "EnsembleCNNModel": CNNExperiment,
    "CNNTCNModel": CNNExperiment,
}

SKLEARN_MODELS = {"SklearnAudioClassifier", "SklearnAudioEnsembleClassifier"}
PYTORCH_MODELS = {"BaselineModel", "LinSeqModel", "CNNModel", "CNNEnsemble", "EnsembleCNNModel", "CNNTCNModel"}


In [33]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def load_yaml(path: str | Path) -> dict:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def read_audio_as_numpy(audio_path: str | Path) -> tuple[np.ndarray, int]:
    """Returns audio as float32 numpy array shaped (T,) or (T, C)."""
    audio_np, sr = sf.read(str(audio_path), dtype=np.float32)
    return audio_np, sr


def ensure_2d_time_channels(audio_np: np.ndarray) -> np.ndarray:
    """Ensure (T, C). If mono (T,), becomes (T, 1)."""
    if audio_np.ndim == 1:
        return audio_np[:, None]
    if audio_np.ndim == 2:
        return audio_np
    raise ValueError(f"Unexpected audio ndim={audio_np.ndim}, shape={audio_np.shape}")


def to_mono_if_requested(audio_tc: np.ndarray, mono: bool) -> np.ndarray:
    """audio_tc is (T, C). If mono=True, returns (T, 1) by averaging channels."""
    if not mono:
        return audio_tc
    return audio_tc.mean(axis=1, keepdims=True).astype(np.float32)


def ensure_stereo(audio_tc: np.ndarray) -> np.ndarray:
    """Ensure (T, 2). If (T,1), duplicate channel. If (T,C>2), take first 2."""
    if audio_tc.shape[1] == 2:
        return audio_tc
    if audio_tc.shape[1] == 1:
        return np.concatenate([audio_tc, audio_tc], axis=1)
    return audio_tc[:, :2]


def to_torch_batch(audio_tc: np.ndarray) -> torch.Tensor:
    """
    Convert (T, C) numpy -> torch (B, C, T) with B=1.
    """
    audio_ct = torch.from_numpy(audio_tc.T.copy())  # (C, T)
    return audio_ct.unsqueeze(0)

def infer_label_from_logits(logits: torch.Tensor) -> int:
    """
    Accepts logits shaped:
    - (B, C) -> one label per clip
    - (B, T, C) -> per-frame logits; vote over frames
    - (T, C) -> treat as single clip; vote
    """
    if logits.ndim == 2:
        # (B, C)
        return int(logits.argmax(dim=-1)[0].item())

    if logits.ndim == 3:
        # (B, T, C)
        per_frame = logits.argmax(dim=-1)[0].to(torch.int64)  # (T,)
        return int(torch.bincount(per_frame).argmax().item())
    if logits.ndim == 2 and logits.shape[0] > 1 and logits.shape[1] > 1:
        per_frame = logits.argmax(dim=-1).to(torch.int64)
        return int(torch.bincount(per_frame).argmax().item())

    raise ValueError(f"Unexpected logits shape: {tuple(logits.shape)}")

In [34]:
def load_model_and_runner(config: dict, ckpt_path: str | Path):
    model_name = config["model"]
    ckpt_path = Path(ckpt_path)

    info = {"model_name": model_name, "ckpt_path": str(ckpt_path)}

    if model_name in SKLEARN_MODELS:
        with ckpt_path.open("rb") as f:
            ckpt = pickle.load(f)

        if model_name == "SklearnAudioEnsembleClassifier":
            base_classifiers = config["ensemble"]["base_classifiers"]
            final_estimator = config["ensemble"]["final_estimator"]
            model = SklearnAudioEnsembleClassifier(
                base_classifiers=base_classifiers,
                final_estimator=final_estimator,
                ensemble_type=config["ensemble"]["type"],
            )
            if "model" in ckpt and ckpt["model"] is not None:
                loaded = ckpt["model"]
                if isinstance(loaded, SklearnAudioEnsembleClassifier):
                    model = loaded  # use fitted wrapper directly
                elif hasattr(loaded, "predict") and hasattr(loaded, "fit"):
                    # if someone saved only the sklearn estimator
                    model.ensemble = loaded
                else:
                    raise TypeError(
                        f"Unsupported ckpt['model'] type: {type(loaded)}. "
                        f"Checkpoint keys: {list(ckpt.keys())}"
                    )
            if "ensemble" in ckpt and ckpt["ensemble"] is not None:
                model.ensemble = ckpt["ensemble"]
            if "pipeline" in ckpt and ckpt["pipeline"] is not None:
                model.pipeline = ckpt["pipeline"]

            def runner(audio_batch_bct: torch.Tensor) -> int:
                if audio_batch_bct.shape[1] != 1:
                    audio_batch_bct = audio_batch_bct.mean(dim=1, keepdim=True)
                features = model.extract_features(audio_batch_bct.cpu())

                if hasattr(model, "ensemble") and model.ensemble is not None:
                    return int(model.ensemble.predict(features)[0])
                if hasattr(model, "pipeline"):
                    return int(model.pipeline.predict(features)[0])

                raise RuntimeError("SklearnAudioEnsembleClassifier has neither 'ensemble' nor 'pipeline' loaded.")

            info["runner_type"] = "sklearn_ensemble"
            return runner, info

        clf_type = str(config["network"]["type"])
        params = config["network"].get("params", {})

        model = SklearnAudioClassifier(
            classifier_type=clf_type,
            sample_rate=int(config["data"]["sample_rate"]),
            n_fft=int(config["network"].get("n_fft", 2048)),
            hop_length=int(config["network"].get("hop_length", 882)),
            n_mels=int(config["network"].get("n_mels", 40)),
            n_estimators=int(params.get("n_estimators", 500)),
            max_depth=int(params.get("max_depth", 20)),
            n_components=int(params.get("n_components", 100)),
            svm_C=float(params.get("svm_C", 10.0)),
            svm_kernel=str(params.get("svm_kernel", "rbf")),
            win_length=int(config["network"].get("win_length", 2048)),
        )

        if "pipeline" in ckpt and ckpt["pipeline"] is not None:
            model.pipeline = ckpt["pipeline"]
        elif "model" in ckpt:
            loaded = ckpt["model"]
            if isinstance(loaded, SklearnAudioClassifier):
                model = loaded
            elif hasattr(loaded, "predict") and hasattr(loaded, "fit"):
                model.pipeline = loaded
            else:
                raise TypeError(
                    f"Unsupported ckpt['model'] type: {type(loaded)}. "
                    f"Checkpoint keys: {list(ckpt.keys())}"
                )
        else:
            raise KeyError(f"Checkpoint missing both 'pipeline' and 'model'. Keys: {list(ckpt.keys())}")


        def runner(audio_batch_bct: torch.Tensor) -> int:
            if audio_batch_bct.shape[1] != 1:
                audio_batch_bct = audio_batch_bct.mean(dim=1, keepdim=True)
            features = model.extract_features(audio_batch_bct.cpu())
            return int(model.pipeline.predict(features)[0])

        info = {
            "ckpt_path": str(ckpt_path),
            "model_name": "SklearnAudioClassifier",
            "runner_type": "sklearn",
            "ckpt_keys": list(ckpt.keys()),
        }
        return runner, info

    if model_name in PYTORCH_MODELS:
        ExpClass = EXPERIMENT_REGISTRY[model_name]
        ModelClass = MODEL_REGISTRY[model_name]

        model_kwargs = dict(config.get("network", {}))
        model_kwargs["sample_rate"] = config["data"]["sample_rate"]
        model_kwargs["n_label"] = config["experiment"]["n_label"]

        if "augmentation" in config:
             model_kwargs.update(config["augmentation"])

        if model_name in {"CNNEnsemble", "EnsembleCNNModel"}:
            model = ModelClass(sample_rate=config["data"]["sample_rate"], **model_kwargs)
        else:
            model = ModelClass(**model_kwargs)

        exp = ExpClass.load_from_checkpoint(
            str(ckpt_path),
            model=model,
            **config["experiment"],
        )
        exp.eval()
        device = exp.device

        def runner(audio_batch_bct: torch.Tensor) -> int:
            with torch.no_grad():
                audio_batch_bct = audio_batch_bct.to(device=device)

                if model_name in {"CNNEnsemble", "EnsembleCNNModel"}:
                    if audio_batch_bct.shape[1] == 1:
                        audio_batch_bct = audio_batch_bct.repeat(1, 2, 1)
                    elif audio_batch_bct.shape[1] > 2:
                        audio_batch_bct = audio_batch_bct[:, :2, :]

                    streams = exp.model.preprocessor(audio_batch_bct)
                    output = exp.model({"streams": streams})
                else:
                    if audio_batch_bct.shape[1] != 1:
                        audio_batch_bct = audio_batch_bct.mean(dim=1, keepdim=True)
                    output = exp.model(audio_batch_bct)

                logits = output["logits"] if isinstance(output, dict) else output
                return infer_label_from_logits(logits)

        info["runner_type"] = "pytorch"
        info["device"] = str(device)
        return runner, info

    raise ValueError(f"Unknown model type: {model_name}")


In [43]:
# ---- User settings ----
CONFIG_PATH = Path("config/cnntcn.yaml")
CKPT_PATH = Path("./../../data/ckpts/cnntcnmodel/epoch=180-val_acc=0.80.ckpt")
TEST_PATH = Path("./../../data/dcase/test/")
VAL_PATH = Path("./../../data/dcase/val/")
GROUPNAME = "fibonacci"
# -----------------------

config = load_yaml(CONFIG_PATH)

runner, info = load_model_and_runner(config, CKPT_PATH)
print("Loaded:", info)

seed for CNNTCNModel: 1500418401


C:\Users\jamil\OneDrive\InfB\.venv\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Loaded: {'model_name': 'CNNTCNModel', 'ckpt_path': '..\\..\\data\\ckpts\\cnntcnmodel\\epoch=180-val_acc=0.80.ckpt', 'runner_type': 'pytorch', 'device': 'cuda:0'}


In [44]:
meta_path = TEST_PATH / "meta_blind.txt"
meta_df = pd.read_csv(meta_path, delimiter="\t", header=None, names=["audio_path"])
n_samples = len(meta_df)
class_labels = np.empty(n_samples, dtype=int)

mono_cfg = bool(config["data"].get("mono", True))
model_name = config["model"]

for i in tqdm(range(n_samples), desc="Blind test inference"):
    rel = meta_df.iloc[i]["audio_path"]
    audio_path = TEST_PATH / rel

    audio_np, sr = read_audio_as_numpy(audio_path)
    audio_tc = ensure_2d_time_channels(audio_np)

    if model_name in {"CNNEnsemble", "EnsembleCNNModel"}:
        audio_tc = ensure_stereo(audio_tc)
    else:
        audio_tc = to_mono_if_requested(audio_tc, mono=mono_cfg)

    audio_batch_bct = to_torch_batch(audio_tc)  # (1, C, T)

    est_label = runner(audio_batch_bct)
    class_labels[i] = est_label


Blind test inference: 100%|██████████| 75/75 [00:02<00:00, 29.77it/s]


In [45]:
out_df = meta_df.copy()
out_df["class_labels"] = class_labels

out_path = TEST_PATH / f"{GROUPNAME}_{model_name}_test1.csv"
out_df.to_csv(out_path, sep="\t", header=None, index=False)
print("Wrote:", out_path)

Wrote: ..\..\data\dcase\test\fibonacci_CNNTCNModel_test1.csv
